# Drug Use Processing

This notebook cleans the EUDA wastewater raw dataset and writes a mirror cleaned file to data/processed.

Output written to data/processed:
- euda_wastewater_ww2026_all_cities_cleaned.csv

In [1]:
from pathlib import Path

import pandas as pd

pd.options.display.max_columns = 200
pd.options.display.float_format = "{:.3f}".format

candidate_roots = [Path.cwd(), Path.cwd().parent]
project_root = None
for root in candidate_roots:
    if (root / "data" / "raw").exists() and (root / "data" / "processed").exists():
        project_root = root
        break

if project_root is None:
    raise FileNotFoundError("Could not resolve the project root with data/raw and data/processed.")

raw_path = project_root / "data" / "raw" / "euda_wastewater_ww2026_all_cities.csv"
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
cleaned_out = processed_dir / "euda_wastewater_ww2026_all_cities_cleaned.csv"

if not raw_path.exists():
    raise FileNotFoundError(f"Missing wastewater raw file: {raw_path}")

print(f"Project root: {project_root}")
print(f"Wastewater raw path: {raw_path}")
print(f"Output cleaned path: {cleaned_out}")

Project root: /home/iuseppe_ares/projects/APPSTAT/MilanCrash
Wastewater raw path: /home/iuseppe_ares/projects/APPSTAT/MilanCrash/data/raw/euda_wastewater_ww2026_all_cities.csv
Output cleaned path: /home/iuseppe_ares/projects/APPSTAT/MilanCrash/data/processed/euda_wastewater_ww2026_all_cities_cleaned.csv


In [2]:
def load_euda_csv(path: Path) -> tuple[pd.DataFrame, str]:
    for encoding in ["utf-8", "utf-8-sig", "latin1"]:
        try:
            return pd.read_csv(path, encoding=encoding), encoding
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError("utf-8", b"", 0, 1, "Could not decode EUDA CSV with tried encodings.")

euda_raw, encoding_used = load_euda_csv(raw_path)
euda_raw.columns = euda_raw.columns.astype(str).str.strip()

print(f"Loaded EUDA rows: {len(euda_raw)} | encoding: {encoding_used}")
print("Columns:", list(euda_raw.columns))
display(euda_raw.head(5))

Loaded EUDA rows: 5238 | encoding: latin1
Columns: ['Year', 'Metabolite', 'Site ID', 'Country', 'City', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday', 'Monday', 'Tuesday', 'Weekday mean', 'Weekend mean', 'Daily mean']


,Year,Metabolite,Site ID,Country,City,Wednesday,Thursday,Friday,Saturday,Sunday,Monday,Tuesday,Weekday mean,Weekend mean,Daily mean
0,2025,MDMA,AT001,AT,Graz,10.820,8.430,8.800,33.560,23.370,13.890,11.540,10.260,19.900,15.770
1,2025,MDMA,AT002,AT,Guntramsdorf,5.620,3.720,3.600,12.770,8.490,4.060,3.670,4.340,7.230,5.990
2,2025,MDMA,AT003,AT,Hall-Wattens,2.070,2.900,2.560,6.150,15.430,8.550,4.270,3.080,8.170,5.990
3,2025,MDMA,AT004,AT,Hallstatt,1.370,1.410,1.770,3.120,1.560,3.550,1.230,1.340,2.500,2.000
4,2025,MDMA,AT006,AT,Innsbruck,8.150,7.880,8.100,10.770,27.410,12.900,7.950,7.990,14.800,11.880


In [3]:
required_cols = [
    "Year",
    "Metabolite",
    "Site ID",
    "Country",
    "City",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
    "Monday",
    "Tuesday",
    "Weekday mean",
    "Weekend mean",
    "Daily mean",
]
missing_cols = [c for c in required_cols if c not in euda_raw.columns]
if missing_cols:
    raise ValueError(f"EUDA CSV is missing expected columns: {missing_cols}")

numeric_cols = [
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
    "Monday",
    "Tuesday",
    "Weekday mean",
    "Weekend mean",
    "Daily mean",
]
string_cols = ["Metabolite", "Site ID", "Country", "City"]

euda_clean = euda_raw.copy()
euda_clean.columns = euda_clean.columns.astype(str).str.strip()

for col in string_cols:
    euda_clean[col] = euda_clean[col].astype("string").str.strip().replace("", pd.NA)

euda_clean["Year"] = pd.to_numeric(euda_clean["Year"], errors="coerce").astype("Int64")
for col in numeric_cols:
    euda_clean[col] = pd.to_numeric(euda_clean[col], errors="coerce")

euda_clean = (
    euda_clean.drop_duplicates()
    .sort_values(["Year", "Country", "City", "Metabolite"], kind="mergesort", na_position="last")
    .reset_index(drop=True)
)

euda_clean.to_csv(cleaned_out, index=False)

qc = pd.Series(
    {
        "rows_cleaned": int(euda_clean.shape[0]),
        "unique_countries": int(euda_clean["Country"].nunique(dropna=True)),
        "unique_cities": int(euda_clean["City"].nunique(dropna=True)),
        "year_min": int(euda_clean["Year"].min()) if euda_clean["Year"].notna().any() else pd.NA,
        "year_max": int(euda_clean["Year"].max()) if euda_clean["Year"].notna().any() else pd.NA,
        "milan_rows": int((euda_clean["City"].str.lower() == "milan").fillna(False).sum()),
        "missing_cells": int(euda_clean.isna().sum().sum()),
    }
)

print(f"Saved: {cleaned_out}")
display(qc.to_frame("value"))
display(euda_clean.head(10))

Saved: /home/iuseppe_ares/projects/APPSTAT/MilanCrash/data/processed/euda_wastewater_ww2026_all_cities_cleaned.csv


,value
rows_cleaned,5238
unique_countries,41
unique_cities,256
year_min,2011
year_max,2025
milan_rows,75
missing_cells,402


,Year,Metabolite,Site ID,Country,City,Wednesday,Thursday,Friday,Saturday,Sunday,Monday,Tuesday,Weekday mean,Weekend mean,Daily mean
0,2011,MDMA,BE002,BE,Antwerp Zuid,29.260,26.160,34.210,85.550,94.150,56.960,38.670,31.370,67.720,52.140
1,2011,amphetamine,BE002,BE,Antwerp Zuid,229.630,211.520,318.890,318.790,311.030,272.620,277.780,239.640,305.330,277.180
2,2011,cannabis,BE002,BE,Antwerp Zuid,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
3,2011,cocaine,BE002,BE,Antwerp Zuid,589.730,610.110,753.980,784.210,919.910,758.490,631.850,610.560,804.150,721.180
4,2011,methamphetamine,BE002,BE,Antwerp Zuid,3.790,4.100,6.150,5.460,6.500,5.830,6.360,4.750,5.990,5.460
5,2011,MDMA,BE007,BE,Brussels,3.540,4.460,3.380,6.010,18.640,8.740,3.540,4.090,8.120,7.450
6,2011,amphetamine,BE007,BE,Brussels,22.920,28.770,39.310,57.570,44.620,32.380,22.180,24.630,43.470,35.390
7,2011,cocaine,BE007,BE,Brussels,146.310,188.310,188.450,236.640,244.040,170.660,115.820,150.150,209.950,184.320
8,2011,methamphetamine,BE007,BE,Brussels,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
9,2011,MDMA,CZ002,CZ,Ceské Budejovice,3.310,1.490,2.780,6.600,3.010,2.470,4.730,3.180,3.710,3.480


## Coverage Profile for Milan (Downstream Readiness)

This summary highlights yearly Milan coverage by metabolite so correlation notebooks can quickly assess sample support.

In [4]:
milan_clean = euda_clean.loc[euda_clean["City"].str.lower() == "milan"].copy()

coverage = (
    milan_clean.groupby(["Year", "Metabolite"], as_index=False)
    .agg(records=("Daily mean", "size"), mean_daily=("Daily mean", "mean"))
    .sort_values(["Year", "Metabolite"])
    .reset_index(drop=True)
)

print("Milan yearly metabolite coverage:")
display(coverage.head(30))

expected_metabolites = ["amphetamine", "cannabis", "cocaine", "mdma", "methamphetamine"]
met_table = (
    coverage.assign(is_present=True)
    .pivot_table(index="Year", columns="Metabolite", values="is_present", aggfunc="max", fill_value=False)
)
for metabolite in expected_metabolites:
    if metabolite not in met_table.columns:
        met_table[metabolite] = False
met_table = met_table[sorted(met_table.columns)]

print("Milan metabolite presence matrix (True means available in that year):")
display(met_table.astype(bool))

years = sorted(milan_clean["Year"].dropna().astype(int).unique())
missing_years = sorted(set(range(min(years), max(years) + 1)) - set(years)) if years else []
print("Year span:", (min(years), max(years)) if years else "no years")
print("Missing years inside span:", missing_years)

Milan yearly metabolite coverage:


,Year,Metabolite,records,mean_daily
0,2011,MDMA,1,7.580
1,2011,amphetamine,1,11.050
2,2011,cannabis,1,27.610
3,2011,cocaine,1,238.850
4,2011,methamphetamine,1,48.680
5,2012,MDMA,1,8.740
6,2012,amphetamine,1,0.000
7,2012,cannabis,1,24.250
8,2012,cocaine,1,241.750
9,2012,methamphetamine,1,10.110


Milan metabolite presence matrix (True means available in that year):


Metabolite,MDMA,amphetamine,cannabis,cocaine,cot,ets,ketamine,mdma,methamphetamine
Year,,,,,,,,,
2011,True,True,True,True,False,False,False,False,True
2012,True,True,True,True,False,False,False,False,True
2013,True,False,True,True,False,False,False,False,True
2014,True,True,False,True,False,False,False,False,True
2015,True,True,False,True,False,False,False,False,True
2016,True,True,False,True,False,False,False,False,True
2017,True,True,False,True,False,False,False,False,True
2018,True,True,True,True,False,False,False,False,True
2019,True,True,True,True,False,False,False,False,True


Year span: (np.int64(2011), np.int64(2025))
Missing years inside span: []
